# How FANTASY decides how many components to use

1. **Build candidates.** `fa.presets.variants("hbeta")` returns 6 models: 1, 2, 3 broad components, each with and without an [OIII]/Hβ outflow component. Broad component *k* gets its own FWHM band so the components cannot swap roles.
2. **Fit each one.** `fit(..., build_up=True)` adds the extra components one at a time, from narrow to wide, and keeps a new component only if χ² drops. This way, adding a component can never make χ² worse.
3. **Rank by BIC** = χ² + k ln N. A more complex model has to buy each extra parameter with a χ² decrease of ln N.
4. **Define 'acceptable'**: χ²_ν ≤ 1.1 × the best χ²_ν. At high S/N no model reaches χ²_ν = 1, and exp(−ΔBIC/2) weights would give all the weight to one model, which hides the model dependence.
5. **Report** the value from the best-BIC model with its statistical error, plus a systematic error equal to the standard deviation over the acceptable models. If only one model is acceptable, sys = NaN: the data clearly pick that model, but the systematic error has not been tested.

In [ ]:
import json, fantasy_agn as fa
spec = fa.desi.from_sparcl(json.load(open("desi/desi_39627963565609635.json"))[1]).to_rest_frame(z=0.18666)
# or: spec = fa.desi.fetch(39627963565609635).to_rest_frame()

In [ ]:
models = fa.presets.variants("hbeta")          # broad_components=(1,2,3), outflow=(False,True)
list(models)

In [ ]:
cmp = fa.compare(spec, models)                  # extra fit options pass through, e.g. errors='mc'
cmp.table[["model", "chi2", "n_free", "reduced_chi2", "dBIC", "acceptable", "Hb FWHM", "Hb FWHM err"]]

In [ ]:
cmp.summary()     # value ± stat_err ± sys_err; min_all / max_all = range over ALL models

Look at the residuals before you trust the decision:

In [ ]:
for name in ["1 broad", cmp.best]:
    cmp.results[name].plot(title=name)

Build your own candidate set, e.g. different broad profiles, or use custom quantities:

In [ ]:
models = {**fa.presets.variants("hbeta", broad_components=(2, 3), outflow=(True,)),
          "2 GH + outflow": fa.presets.hbeta(broad_components=2, outflow=True, broad_profile="gauss-hermite")}
q = {"[OIII] FWHM": lambda r: (r.lines.set_index("line").loc["OIII5007", "fwhm"],
                               r.lines.set_index("line").loc["OIII5007", "fwhm_err"])}
fa.compare(spec, models, quantities=q).summary()